In [ ]:
import scanpy as sc
import pandas as pd
from matplotlib import pylab
import random
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
import itertools
import plotly.express as px
import yaml


import numpy as np
import random
import anndata as ad
from scipy.sparse import  csr_matrix, issparse
from scipy import sparse

from matplotlib.colors import TwoSlopeNorm

import scanpy.external as sce
import sys



In [ ]:
import anndata

In [ ]:

sc.settings.set_figure_params(dpi=80, facecolor='white', dpi_save=500)
pylab.rcParams['figure.figsize'] = (6, 6)
homeDir = os.getenv("HOME")
sys.path.insert(1, homeDir+"/utils/")

from PlotPCA_components import *
from AtlasClasses import *
DS="hormones_substudy"
ReferenceTissue="cortex"

with open(homeDir+"/utils/ReferenceDict.yaml", 'r') as file:
    ReferencePaths = yaml.safe_load(file)
    for k in list(ReferencePaths.keys()):
        ReferencePaths[k]["adataPath"] = "/group/testa/Users/davide.castaldi/Polaroids_spinoff"+ReferencePaths[k]["adataPath"]
        #ReferencePaths[k]["signaturePath"] = homeDir+ReferencePaths[k]["signaturePath"]
        ReferencePaths[k]["signaturePath"] = "/group/testa/Users/davide.castaldi/Polaroids_spinoff"+ReferencePaths[k]["signaturePath"]
        ReferencePaths[k]["signaturePurityPath"] = "/group/testa/Users/davide.castaldi/Polaroids_spinoff"+ReferencePaths[k]["signaturePurityPath"]


ReferencePaths


# Import

In [ ]:
adata = sc.read_h5ad(homeDir+"/adatas/6.1.Annotated.{}.h5ad".format(DS))
# adata.write_h5ad("./adatas/6.1.Annotated.{}.h5ad".format(DS))

# adata.layers["logNorm"] = adata.X.copy()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_stacked_fractions_by_cluster(
	Combined,
	variable1: str,                 # e.g. "MacroGroups"   (cell type / hue) — sums to 1 per replicate
	variable2: str,                 # e.g. "individual"    (replicates)
	variable3: str | None = None,   # optional group to order by OR aggregate by
	*,
	shrink = .8,
	# Ordering (used when NOT aggregating)
	order_agg: str = "mean",        # for numeric variable3: "mean" or "median"
	ascending: bool = False,        # numeric sort direction for variable3 order
	# ---- NEW: explicit x-order for variable2 bars ----
	x_order_v2: list[str] | None = None,     # user-defined order for variable2 bars
	append_unlisted_v2: bool = True,         # if True, append any variable2 not in x_order_v2
	# Aggregation toggle
	aggregate_by_variable3: bool = False,  # if True, x-axis is variable3 and heights are medians across replicates
	renormalize_medians: bool = True,      # after taking medians per cell type, renormalize each bar to sum to 1
	# Plot look
	figsize=(20, 5), linewidth=.1,
	dpi=100,
	palette_name: str = "Set3",     # used only if Combined.uns colors are missing
	random_seed: int | None = 123,
	edgecolor: str = "black",       # visible edges for tiny linewidths
	# ---- Annotation ribbons ----
	annotations: list[str] | None = None,               # obs columns -> colored ribbons above bars
	annotation_palettes: dict[str, dict] | None = None, # optional {annotation: {value: color}}
	annotation_palette_names: dict[str, str] | None = None,  # optional {annotation: seaborn palette name}
	annotation_height: float = 0.02,
	annotation_gap: float = 0.01,
	annotation_offset: float = 0.01,
	annotation_box_width: float = 1.0,
	# ---- Second-row manual legend strip (as subplots) ----
	legend_row_height: float = 2.2,   # height (relative) for the 2nd row
	legend_hspace: float = 0.25,      # distance between main plot and legend-row
	legend_dot_radius: float = 0.05,
	legend_fontsize: int = 9,
	legend_title_fontsize: int = 12,
	# ---- NEW: tunable folding (columns) for legend circles ----
	legend_cols_v1: int = 2,                         # columns for variable1 legend panel
	legend_cols_annotations: int | dict[str,int] = 2, # int for all, or per-annotation dict
	# ---- NEW: tunable spacing inside each legend panel (rows/cols grid) ----
	legend_grid_left: float = 0.08,     # axes coords; smaller -> more left margin
	legend_grid_right: float = 0.92,    # axes coords; larger -> more right margin
	legend_grid_top: float = 0.82,      # top y for first row
	legend_grid_bottom: float = 0.10,   # bottom y for last row
	legend_col_compact: float = 1.0,    # 0<factor<=1; <1 squeezes columns toward panel center
	legend_row_compact: float = 1.0,    # 0<factor<=1; <1 squeezes rows toward panel center
	legend_text_dx: float = 0.06,       # x offset from circle to label (axes coords)
	# ---- NEW: accept external axis ----
	ax=None,                            # pass an Axes to draw into; legend panels are skipped in this mode
):
	"""
	Row 1: main stacked bars (single axis)
	Row 2: (only when ax is None) PANELS (equal widths): panel 0 = variable1 legend; panels 1..K = per-annotation.
		   Inside each panel, values are colored circles arranged in an N-COLUMN grid (tunable).
		   No built-in legend API is used.

	Subordering: within each `variable3` bucket (when not aggregating), replicates are sub-ordered by
	the provided annotations in order (ann1 -> ann2 -> ...). Primary order source remains `variable3`.

	NEW:
	- You can force the x-axis order for variable2 bars via x_order_v2 (only when aggregate_by_variable3=False).
	"""
	import numpy as np
	import pandas as pd
	import seaborn as sns
	import matplotlib.pyplot as plt
	import matplotlib.patches as mpatches
	from matplotlib.gridspec import GridSpec

	rng = np.random.default_rng(random_seed) if random_seed is not None else None
	annotations = list(annotations) if annotations else []

	# ---------- helpers for DISTINCT colors (avoid repetition) ----------
	def _distinct_colors(n: int, pref_name: str | None = None):
		try:
			if pref_name is not None:
				cols = sns.color_palette(pref_name, n_colors=n)
				if len({tuple(c) for c in cols}) == n:
					return cols
		except Exception:
			pass
		return sns.husl_palette(n, s=0.8, l=0.5)

	def _mode_val(s):
		m = s.mode()
		return str(m.iat[0] if not m.empty else s.iloc[0])

	# ---------- base dataframe ----------
	obs = Combined.obs.copy()
	keep = obs[variable2].notna()
	if variable1 in obs.columns:
		keep &= obs[variable1].notna()

	cols = [variable1, variable2] + ([variable3] if (variable3 is not None and variable3 in obs.columns) else [])
	for a in annotations:
		if a in obs.columns and a not in cols:
			cols.append(a)

	plotDF = obs.loc[keep, cols].copy()
	plotDF[variable1] = plotDF[variable1].astype(str)
	plotDF[variable2] = plotDF[variable2].astype(str)
	if variable3 is not None and variable3 in plotDF.columns:
		plotDF[variable3] = plotDF[variable3].astype(str)
	for a in annotations:
		if a in plotDF.columns:
			plotDF[a] = plotDF[a].astype(str)

	# ---------- compositions (per replicate variable2) ----------
	comp = (
		plotDF.groupby([variable1, variable2])
			  .size()
			  .reset_index(name="number_of_cells")
	)
	comp["clusterTotal"] = comp.groupby(variable2)["number_of_cells"].transform("sum")
	comp["CellsFraction"] = comp["number_of_cells"] / comp["clusterTotal"]

	# ---------- palette & hue order ----------
	palette = None
	hue_order = None
	try:
		colors_key = f"{variable1}_colors"
		if colors_key in Combined.uns:
			if pd.api.types.is_categorical_dtype(Combined.obs[variable1]):
				colors = Combined.uns[colors_key]
				cats = list(Combined.obs[variable1].cat.categories)
				cats_str = [str(c) for c in cats]
				palette = {c: col for c, col in zip(cats_str, colors)}
				hue_order = cats_str
				print(f"using uns colors for '{variable1}'")
	except Exception:
		palette = None
		pass

	if palette is None:
		print("Assigning palette manually.")
		present_levels = sorted(comp[variable1].astype(str).unique().tolist())
		_cols = _distinct_colors(len(present_levels), pref_name=palette_name)
		palette = {lev: col for lev, col in zip(present_levels, _cols)}
		hue_order = present_levels
		present = set(comp[variable1].unique().tolist())
		hue_order = [h for h in hue_order if h in present]

	# ---- helper to build annotation color maps (distinct within each annotation) ----
	def _make_annotation_color_maps(df_levels: pd.DataFrame) -> dict:
		out = {}
		if not annotations:
			return out
		default_cycle = ["tab20", "Set2", "Dark2", "Paired", "tab20b", "tab20c"]
		name_map = annotation_palette_names or {}
		for i, a in enumerate(annotations):
			if a not in df_levels.columns:
				out[a] = {}
				continue
			vals = df_levels[a].dropna().astype(str).unique().tolist()
			if len(vals) == 0:
				out[a] = {}
				continue
			if annotation_palettes and a in annotation_palettes and annotation_palettes[a]:
				user_map = annotation_palettes[a].copy()
				missing = [v for v in vals if v not in user_map]
				if missing:
					pref = name_map.get(a, default_cycle[i % len(default_cycle)])
					extra_cols = _distinct_colors(len(missing), pref_name=pref)
					for v, c in zip(missing, extra_cols):
						user_map[v] = c
				if len({tuple(c) for c in user_map.values()}) < len(user_map):
					fresh = _distinct_colors(len(vals), pref_name=None)
					user_map = {v: c for v, c in zip(vals, fresh)}
				out[a] = user_map
			else:
				pref = name_map.get(a, default_cycle[i % len(default_cycle)])
				cols_here = _distinct_colors(len(vals), pref_name=pref)
				out[a] = {v: c for v, c in zip(vals, cols_here)}
		return out

	# ---------- ordering for variable2 (non-aggregated branch) ----------
	x_order = None

	# 0) explicit user-provided x-order takes precedence (only when NOT aggregating)
	if (not aggregate_by_variable3) and (x_order_v2 is not None):
		seen = set()
		base = []
		for x in x_order_v2:
			s = str(x)
			if s not in seen:
				seen.add(s)
				base.append(s)
		if append_unlisted_v2:
			extras = [s for s in comp[variable2].astype(str).unique().tolist() if s not in seen]
			base = base + sorted(extras)
		x_order = base

	# 1) otherwise, try to derive x_order from variable3 (your existing logic)
	elif variable3 is not None and variable3 in plotDF.columns:
		tmp = plotDF[[variable2, variable3]].dropna()

		if pd.api.types.is_numeric_dtype(tmp[variable3]):
			agg_fn = {"mean": "mean", "median": "median"}.get(str(order_agg).lower(), "mean")
			base_order = (
				tmp.groupby(variable2)[variable3]
				   .agg(agg_fn)
				   .sort_values(ascending=ascending)
			)

			if annotations:
				ann_modes_df = plotDF.groupby(variable2).agg({a: _mode_val for a in annotations if a in plotDF.columns}).astype(str)
				ann_rank = {}
				for a in annotations:
					if a not in plotDF.columns:
						continue
					try:
						s = Combined.obs[a]
						if pd.api.types.is_categorical_dtype(s) and s.dtype.ordered:
							levels = [str(x) for x in s.cat.categories]
						else:
							levels = sorted(plotDF[a].astype(str).unique().tolist())
					except Exception:
						levels = sorted(plotDF[a].astype(str).unique().tolist())
					ann_rank[a] = {v: i for i, v in enumerate(levels)}

				reps = list(base_order.index.astype(str))
				num_vals = base_order.astype(float).to_dict()

				def _ann_key(rep):
					keys = []
					for a in annotations:
						if a in ann_rank and rep in ann_modes_df.index:
							val = str(ann_modes_df.loc[rep, a])
							keys.append(ann_rank[a].get(val, len(ann_rank[a])))
						else:
							keys.append(float("inf"))
					keys.append(rep)
					return tuple(keys)

				# respect ascending/descending even with annotations
				def _num_primary(v):
					if not np.isfinite(v):
						return np.inf
					return v if ascending else -v

				reps.sort(key=lambda r: (_num_primary(float(num_vals.get(r, np.inf))), _ann_key(r)))
				x_order = reps
			else:
				x_order = base_order.index.astype(str).tolist()

		else:
			v3 = tmp[variable3]
			if pd.api.types.is_categorical_dtype(v3) and v3.dtype.ordered:
				v3_levels = [str(x) for x in v3.cat.categories]
			else:
				v3_levels = sorted(v3.astype(str).unique().tolist())

			dominant = (
				tmp.groupby([variable2, variable3])
				   .size()
				   .reset_index(name="n")
				   .sort_values("n", ascending=False)
				   .drop_duplicates(subset=[variable2])
			)
			dominant[variable3] = dominant[variable3].astype(str)

			reps_by_v3 = {lvl: [] for lvl in v3_levels}
			for _, r in dominant.iterrows():
				reps_by_v3[str(r[variable3])].append(str(r[variable2]))

			ann_rank = {}
			ann_modes_df = None
			if annotations:
				for a in annotations:
					if a not in plotDF.columns:
						continue
					try:
						s = Combined.obs[a]
						if pd.api.types.is_categorical_dtype(s) and s.dtype.ordered:
							levels = [str(x) for x in s.cat.categories]
						else:
							levels = sorted(plotDF[a].astype(str).unique().tolist())
					except Exception:
						levels = sorted(plotDF[a].astype(str).unique().tolist())
					ann_rank[a] = {v: i for i, v in enumerate(levels)}
				ann_modes_df = plotDF.groupby(variable2).agg({a: _mode_val for a in annotations if a in plotDF.columns}).astype(str)

			for lvl in v3_levels:
				reps = reps_by_v3[lvl]
				if annotations and ann_modes_df is not None:
					def _key(rep):
						keys = []
						for a in annotations:
							if a in ann_rank and rep in ann_modes_df.index:
								val = str(ann_modes_df.loc[rep, a])
								keys.append(ann_rank[a].get(val, len(ann_rank[a])))
							else:
								keys.append(float("inf"))
						keys.append(rep)
						return tuple(keys)
					reps.sort(key=_key)
				else:
					reps.sort()
				reps_by_v3[lvl] = reps

			x_order = [rep for lvl in v3_levels for rep in reps_by_v3[lvl]]

	# apply ordering to comp (variable2)
	if x_order and (not aggregate_by_variable3):
		comp[variable2] = pd.Categorical(comp[variable2].astype(str), categories=x_order, ordered=True)

	# ---------- build plot_data ----------
	if aggregate_by_variable3:
		if variable3 is None or variable3 not in plotDF.columns:
			raise ValueError("aggregate_by_variable3=True requires variable3 to be provided and present in Combined.obs")

		# map replicate -> variable3 group (mode per replicate)
		rep_to_group = (
			plotDF[[variable2, variable3]]
			.dropna()
			.groupby(variable2)[variable3]
			.agg(_mode_val)
		)

		comp_g = comp.merge(rep_to_group.rename(variable3), left_on=variable2, right_index=True, how="left")
		comp_g = comp_g.dropna(subset=[variable3])
		comp_g[variable3] = comp_g[variable3].astype(str)

		# median across replicates for each group and celltype
		plot_data = (
			comp_g.groupby([variable3, variable1])["CellsFraction"]
				  .median()
				  .reset_index()
		)

		if renormalize_medians:
			tot = plot_data.groupby(variable3)["CellsFraction"].transform("sum")
			plot_data["CellsFraction"] = plot_data["CellsFraction"] / tot.replace(0, np.nan)

		# make variable3 categorical order if possible
		try:
			s3 = Combined.obs[variable3]
			if pd.api.types.is_categorical_dtype(s3) and s3.dtype.ordered:
				v3_levels = [str(x) for x in s3.cat.categories]
			else:
				v3_levels = sorted(plot_data[variable3].astype(str).unique().tolist())
		except Exception:
			v3_levels = sorted(plot_data[variable3].astype(str).unique().tolist())

		plot_data[variable3] = pd.Categorical(plot_data[variable3].astype(str), categories=v3_levels, ordered=True)

	else:
		plot_data = comp

	# ---------- annotation color maps ----------
	if annotations:
		color_maps = _make_annotation_color_maps(plotDF[[a for a in annotations if a in plotDF.columns]].copy())
	else:
		color_maps = {}

	# ---------- FIGURE LAYOUT ----------
	own_axes = ax is None
	if own_axes:
		n_panels = 1 + len(annotations)
		width_ratios = [1] * n_panels
		fig_h = figsize[1] + legend_row_height + 0.3
		fig = plt.figure(figsize=(figsize[0], fig_h), dpi=dpi)
		gs = GridSpec(
			nrows=2,
			ncols=n_panels,
			height_ratios=[10, legend_row_height],
			width_ratios=width_ratios,
			figure=fig
		)
		gs.update(hspace=legend_hspace)
		ax = fig.add_subplot(gs[0, :])
	else:
		fig = ax.figure

	# ---------- MAIN STACKED BARS ----------
	if aggregate_by_variable3:
		sns.histplot(
			plot_data,
			x=variable3,
			hue=variable1,
			weights="CellsFraction",
			multiple="stack",
			shrink=shrink,
			palette=palette,
			hue_order=hue_order,
			linewidth=linewidth,
			edgecolor=edgecolor,
			legend=False,
			ax=ax,
		)
		ax.set_ylabel("CellsFraction (median across replicates)")
		xcats = list(plot_data[variable3].cat.categories) if isinstance(plot_data[variable3].dtype, pd.CategoricalDtype) else sorted(plot_data[variable3].astype(str).unique().tolist())

	else:
		sns.histplot(
			plot_data,
			x=variable2,
			hue=variable1,
			weights="CellsFraction",
			multiple="stack",
			shrink=shrink,
			palette=palette,
			hue_order=hue_order,
			linewidth=linewidth,
			edgecolor=edgecolor,
			legend=False,
			ax=ax,
		)
		ax.set_ylabel("CellsFraction")
		if isinstance(plot_data[variable2].dtype, pd.CategoricalDtype):
			xcats = list(plot_data[variable2].cat.categories)
		else:
			xcats = sorted(plot_data[variable2].astype(str).unique().tolist())

	ax.set(ylim=(0, 1))
	sns.despine(ax=ax)
	ax.grid(False)
	ax.tick_params(axis="x", rotation=90)

	# ---------- RIBBONS ABOVE BARS ----------
	if annotations:
		xticks = ax.get_xticks()
		x_map = {c: xticks[i] for i, c in enumerate(xcats)} if len(xticks) >= len(xcats) else {c: i for i, c in enumerate(xcats)}
		trans = ax.get_xaxis_transform()
		base_y = 1.0 + annotation_offset

		if aggregate_by_variable3:
			# mode annotation per replicate, then mode per group (variable3)
			v2_v3 = (
				plotDF[[variable2, variable3]]
				.dropna()
				.groupby(variable2)[variable3]
				.agg(_mode_val)
			)

			mode_per_rep = (
				plotDF.groupby(variable2).agg({a: _mode_val for a in annotations if a in plotDF.columns}).astype(str)
				.join(v2_v3.rename("group"))
			)

			mode_per_group = (
				mode_per_rep.groupby("group").agg({a: _mode_val for a in annotations if a in mode_per_rep.columns}).reset_index()
				.rename(columns={"group": variable3})
			)

			for _, row in mode_per_group.iterrows():
				cat = str(row[variable3])
				if cat not in x_map:
					continue
				x_center = x_map[cat]
				x0 = x_center - annotation_box_width / 2
				for i, a in enumerate(annotations):
					val = str(row.get(a, ""))
					col = color_maps.get(a, {}).get(val, (0.9, 0.9, 0.9))
					ax.add_patch(
						mpatches.Rectangle(
							(x0, base_y + i * (annotation_height + annotation_gap)),
							annotation_box_width, annotation_height,
							color=col, transform=trans, clip_on=False
						)
					)

		else:
			mode_per_rep = plotDF.groupby(variable2).agg({a: _mode_val for a in annotations if a in plotDF.columns}).reset_index()
			for _, row in mode_per_rep.iterrows():
				cat = str(row[variable2])
				if cat not in x_map:
					continue
				x_center = x_map[cat]
				x0 = x_center - annotation_box_width / 2
				for i, a in enumerate(annotations):
					val = str(row.get(a, ""))
					col = color_maps.get(a, {}).get(val, (0.9, 0.9, 0.9))
					ax.add_patch(
						mpatches.Rectangle(
							(x0, base_y + i * (annotation_height + annotation_gap)),
							annotation_box_width, annotation_height,
							color=col, transform=trans, clip_on=False
						)
					)

	# ---------- SECOND ROW LEGEND PANELS (only if we own the axes) ----------
	if own_axes:
		def _style_mini_axis(a):
			a.set_axis_off()
			a.set_xlim(0, 1)
			a.set_ylim(0, 1)

		def _dot(axmini, x, y, color):
			circ = mpatches.Circle((x, y), radius=legend_dot_radius, facecolor=color, edgecolor="black", linewidth=.5)
			axmini.add_patch(circ)

		n_panels = 1 + len(annotations)
		bottom_axes = [fig.add_subplot(gs[1, i]) for i in range(n_panels)]

		def _draw_panel(axmini, title, labels, colors, ncols_desired: int):
			_style_mini_axis(axmini)
			axmini.text(
				0.5, 0.98, str(title),
				transform=axmini.transAxes, ha="center", va="top",
				fontsize=legend_title_fontsize
			)
			labels = list(labels)
			n = len(labels)
			if n == 0:
				return
			ncols = max(1, min(ncols_desired, n))
			nrows = int(np.ceil(n / ncols))

			def _squash_interval(lo, hi, factor):
				mid = 0.5
				return mid - (mid - lo) * factor, mid + (hi - mid) * factor

			x_left, x_right = _squash_interval(legend_grid_left, legend_grid_right, legend_col_compact)
			y_top, y_bottom = _squash_interval(legend_grid_top, legend_grid_bottom, legend_row_compact)

			xs = np.linspace(x_left, x_right, num=ncols)
			ys = np.array([0.45]) if nrows == 1 else np.linspace(y_top, y_bottom, num=nrows)

			k = 0
			for r in range(nrows):
				for c in range(ncols):
					if k >= n:
						break
					xv, yv = xs[c], ys[r]
					lab = labels[k]
					col = colors[lab]
					_dot(axmini, xv, yv, col)
					axmini.text(xv + legend_text_dx, yv, str(lab), va="center", ha="left",
								fontsize=max(legend_fontsize - 1, 6))
					k += 1

		colors_v1 = {lab: palette[lab] for lab in hue_order}
		_draw_panel(bottom_axes[0], f"{variable1} | Main compositions", hue_order, colors_v1, legend_cols_v1)

		if annotations:
			for j, a_name in enumerate(annotations, start=1):
				vals = list(color_maps.get(a_name, {}).keys())
				cols_this = (
					legend_cols_annotations[a_name]
					if isinstance(legend_cols_annotations, dict) and a_name in legend_cols_annotations
					else (legend_cols_annotations if isinstance(legend_cols_annotations, int) else 2)
				)
				_draw_panel(bottom_axes[j], f"Annotation {j} | {a_name}", vals, color_maps.get(a_name, {}), cols_this)

	# ---------- return ----------
	info = {
		"x_axis": "variable3" if aggregate_by_variable3 else "variable2",
		"x_order": (
			list(plot_data[variable3].cat.categories) if aggregate_by_variable3 and isinstance(plot_data[variable3].dtype, pd.CategoricalDtype)
			else (list(plot_data[variable2].cat.categories) if (not aggregate_by_variable3) and isinstance(plot_data[variable2].dtype, pd.CategoricalDtype) else None)
		),
		"hue_order": hue_order,
		"palette": palette,
		"edgecolor": edgecolor,
		"annotations": {a: {"colors": (color_maps.get(a, {}) if annotations else {})} for a in annotations} if annotations else {},
		"legend_layout": {
			"rendered": bool(own_axes),
			"panels": (1 + len(annotations)) if own_axes else 0,
			"equal_widths": True if own_axes else None,
			"row_height": legend_row_height if own_axes else None,
			"hspace": legend_hspace if own_axes else None,
			"folding": {
				"variable1_cols": legend_cols_v1,
				"annotation_cols": legend_cols_annotations
			} if own_axes else None,
			"titles_centered": True if own_axes else None,
			"grid": {
				"left": legend_grid_left, "right": legend_grid_right,
				"top": legend_grid_top, "bottom": legend_grid_bottom,
				"col_compact": legend_col_compact, "row_compact": legend_row_compact,
				"text_dx": legend_text_dx
			} if own_axes else None
		},
	}
	return fig, ax, info

In [ ]:
adata.obs["Major_celltype"] = adata.obs["Leiden_1"].replace(pd.crosstab(adata.obs["MacroCelltype"], adata.obs["Leiden_1"]).idxmax(0).to_dict())

In [ ]:
adata.obs["line_condition_replicate"] = adata.obs["line"].astype(str)+ "_" +adata.obs["condition"].astype(str)+"_Replicate" + adata.obs["replicate"].astype(str)

In [ ]:
import os
import yaml
import pandas as pd

os.makedirs("./figures/", exist_ok=True)

adata.obs["line_condition_replicate"] = (
    adata.obs["line"].astype(str)
    + "_"
    + adata.obs["condition"].astype(str)
    + "_Replicate"
    + adata.obs["replicate"].astype(str)
)

with open("./resources/colorMaps.yaml", "r") as f:
    sample_id_cmap = yaml.load(f, Loader=yaml.FullLoader)["sample_id"]

for line in adata.obs["line"].unique():

    adataLine = adata[adata.obs["line"] == line].copy()

    sample_meta = (
        adataLine.obs[
            ["sample_id", "line_condition_replicate", "condition", "replicate", "line"]
        ]
        .drop_duplicates()
        .copy()
    )

    sample_meta["replicate"] = sample_meta["replicate"].astype(int)

    sample_order = (
        sample_meta
        .sort_values(["condition", "replicate", "line_condition_replicate"])
        ["line_condition_replicate"]
        .astype(str)
        .tolist()
    )

    adataLine.obs["line_condition_replicate"] = pd.Categorical(
        adataLine.obs["line_condition_replicate"].astype(str),
        categories=sample_order,
        ordered=True,
    )

    # rename YAML keys: sample_id -> line_condition_replicate
    sample_id_to_lcr = (
        sample_meta
        .assign(
            sample_id=lambda x: x["sample_id"].astype(str),
            line_condition_replicate=lambda x: x["line_condition_replicate"].astype(str),
        )
        .set_index("sample_id")["line_condition_replicate"]
        .to_dict()
    )

    line_condition_replicate_cmap = {
        sample_id_to_lcr[sample_id]: color
        for sample_id, color in sample_id_cmap.items()
        if sample_id in sample_id_to_lcr
    }

    adataLine.uns["line_condition_replicate_colors"] = [
        line_condition_replicate_cmap[c]
        for c in adataLine.obs["line_condition_replicate"].cat.categories
    ]

    fig, ax, info = plot_stacked_fractions_by_cluster(
        adataLine,
        variable1="line_condition_replicate",
        variable2="Leiden_1",
        variable3="Major_celltype",
        annotations=["Major_celltype"],
        aggregate_by_variable3=False,
        figsize=(30, 10),
        linewidth=0,
        legend_hspace=1,
        legend_row_height=10,
        shrink=.95,legend_cols_v1=3,
        legend_cols_annotations=1,
        legend_fontsize=10,
        legend_title_fontsize=15,
        renormalize_medians=False,
    )

    fig.savefig(f"./figures/stacked_clustersComposition_{line}.svg", bbox_inches="tight")
    del adataLine

# Individual plots per line-condition **Progenitors**

In [ ]:
dir_path = homeDir+"/adatas/{}_ByMacroCelltype".format(DS)
celltype="Progenitors"
adata = sc.read_h5ad(dir_path+"/{}.{}.LineMagnification.h5ad".format(DS,celltype))
def sanitize_strings(value):
    if isinstance(value, str):  # Only process string values
        return (
            value.replace('_', '')  # Replace underscores with spaces
            .replace('-', '')      # Replace hyphens with spaces
            .replace('@', '')       # Example: Remove "@" character
        )
    return value  # Leave non-strings untouched
pcs=15
n_neighbors=30
control = "DMSO"
prop = .2

colorDict = {"3PBA":["#FFFFFF","#9D9D9C","#1D1D1B"],
"DPHP":["#FFFFFF","#E3D5EA","#8F4A97"],
"BPA":["#FFFFFF","#D8AF59","#6B3010"],
"MEP":["#FFFFFF","#E1AED1","#B81A5D"],
"BPF":["#FFFFFF","#FFED00","#FBBA00"],
"MBzP":["#FFFFFF","#B9E0E4","#2C509F"],
"TCP":["#FFFFFF","#D3D800","#009640"],
"MIX":["#FFFFFF","#F28F83","#E52423"]}

sc.pl.umap(adata, color=["line","condition"], size=10, vmin='p1', vmax='p99')
# HVGs per line and contrast (agonist vs antagonist)

Compound = list(set(["_".join(i.split("_")[:-1]) for i in adata.obs["condition"].unique().tolist() if i != "DMSO"]+["DMSO"]))
for compound in Compound:
    adata.obs.loc[adata.obs["condition"].str.contains(compound),"compound"] = compound
adata.obs["compound"].value_counts().sum()
adata.obs["line_compound"] = adata.obs["line"].astype(str) + "_" + adata.obs["compound"].astype(str)
adata.obs["line_condition"] = adata.obs["line"].astype(str) + "_" + adata.obs["condition"].astype(str)
ReverseMappingDict = {sanitize_strings(i):i for i in  adata.obs["line_compound"].unique().tolist()}
ReverseMappingDict
HVGSCondict = {}
for compound in adata.obs["line_condition"].unique().tolist():
    adatacompound = adata[adata.obs["line_condition"] == compound]

    nsamples = len(adatacompound.obs["sample_id"].unique().tolist())
    print(f"Found {nsamples} repliacates for  {compound}")
    if nsamples < 3:
        minOverlap = nsamples
    else:
        minOverlap = nsamples-1
    print(f"Only HVGs common to {minOverlap} replicates will be kept")

    sc.pp.highly_variable_genes(adatacompound, batch_key="sample_id", flavor="seurat", n_top_genes=2000)
    CommonHVGs = set(adatacompound.var_names[adatacompound.var["highly_variable_nbatches"] >= minOverlap])
    nHVGs = len(CommonHVGs)
    print(f"Found {nHVGs} common HVGs across {minOverlap} replicates for  exposure {compound} ")
    print("\n")
    HVGSCondict[compound] = CommonHVGs
HVGSCondlineDict = {}
for compound in adata.obs["line_compound"].unique().tolist():

    adataCond = adata[adata.obs["line_compound"] == compound].copy()
    sc.pp.highly_variable_genes(adataCond, flavor="seurat", n_top_genes=2000)
    HVGSLine = set(adataCond.var_names[adataCond.var["highly_variable"]].tolist())
    nHVGs = len(HVGSLine)
    print(f"Found {nHVGs} common HVGs across for  exposure {compound} ")
    print("\n")
    HVGSCondlineDict[compound] = HVGSLine



IntersectionDEGs = {}
for k in [K for K in list(HVGSCondlineDict.keys()) if control not in K]:
    crossHVGs = HVGSCondlineDict[k]
    intraHVGs = HVGSCondict["{}_AGONIST".format(k)].union(HVGSCondict["{}_ANTAGONIST".format(k)])
    print("Found {} intra compound HVGs ".format(len(intraHVGs)))
    OverallHVGs = list(crossHVGs.union(intraHVGs))
    print("Found {} final HVGs for {}".format(len(OverallHVGs), k))
    IntersectionDEGs[k] = OverallHVGs

# Divide by line_compound and process each
IntegratedDict = {}
designDict = {}
for line_compound in [i for i in adata.obs["line_compound"].unique().tolist()if "DMSO" not in i]:
    print(f"Processing {line_compound}")
    adataCondition = adata[adata.obs["line_compound"] == line_compound].copy()
    print(adataCondition.shape)
    print(adataCondition.obs["line"].unique().tolist())
    print(adataCondition.obs["condition"].unique().tolist())
    adataCondition.var["highly_variable"] = adataCondition.var_names.isin(IntersectionDEGs[line_compound])
    print(adataCondition.var["highly_variable"].sum())

    sc.tl.pca(adataCondition, use_highly_variable=True)
    sc.pp.neighbors(adataCondition, n_pcs=pcs, n_neighbors=n_neighbors)
    sc.tl.umap(adataCondition)

    sc.pl.umap(adataCondition, color=["Consensus_call","replicate","condition"], size=40, vmin='p1', vmax='p99', ncols=3,wspace=.4)
    #IntegratedDict[condition] = adataCondition

    # And prepare each anndata for milo
    ParsedObs = adataCondition.obs[["sample_id","line_compound","line","condition","Consensus_call","replicate"]].copy()
    ParsedObs = ParsedObs.applymap(sanitize_strings).loc[adataCondition.obs_names]
    adataCondition.obs = ParsedObs.copy()
    adataCondition.obsp = None
    del adataCondition.uns["neighbors"]
    adataCondition.obs["MiloSlicer"] = sanitize_strings(line_compound)
    del adataCondition.layers
    del adataCondition.varm
    #Store anndata
    IntegratedDict[sanitize_strings(line_compound)] = adataCondition

    # And its design matrix
    design_df = adataCondition.obs.copy()
    design_df.drop_duplicates(inplace=True)
    design_df.index = design_df['sample_id']
    design_df.index.name = None
    print("The following design matrix will be used")
    print(design_df)
    design_df["MiloSlicer"] = sanitize_strings(line_compound)
    designDict[sanitize_strings(line_compound)] = design_df

CombinedDesign = pd.concat(list(designDict.values()))
CombinedAdatas = ad.concat(list(IntegratedDict.values()))

In [ ]:
tmp = sc.read_h5ad("/group/testa/Common/scRNAseq/visualization/ENDPOINTS_sc/{}.{}.LineMagnification.h5ad".format(DS,celltype))
sc.pl.umap(tmp, color=["line","condition"], size=10, vmin='p1', vmax='p99')

In [ ]:
colorDict = {'Depleted':"#534ca3", 'Enriched':"#be1622", 'NotEnriched':"white"}

for line_compound in list(IntegratedDict.keys()):
    adataCondition = IntegratedDict[line_compound]
    parsedContrast = (adataCondition.obs["MiloSlicer"].str.replace("CTL04E", "CTL04E_")\
        .str.replace("CTL08A", "CTL08A_")+"_AGONIST_DA").str.replace("ARYLHYD", "ARYL_HYD")\
            .str.replace("LIVERX", "LIVER-X").unique()[0]
    adataCondition.obs[parsedContrast] =  tmp.obs.loc[adataCondition.obs_names, parsedContrast]
    unscolors = [colorDict[cat] for cat in adataCondition.obs[parsedContrast].astype("category").cat.categories.tolist()]
    adataCondition.uns[f"{parsedContrast}_colors"] = unscolors
    sc.pl.umap(adataCondition, color=["Consensus_call","replicate","condition",parsedContrast], size=40, vmin='p1',
     vmax='p99', ncols=4,wspace=.6, save=f"_{line_compound}_{celltype}_withContrast.png", add_outline=True,outline_width=(0.1, 0.01))

# Individual plots per line-condition **Neurons**

In [ ]:
dir_path = homeDir+"/adatas/{}_ByMacroCelltype".format(DS)
celltype="Neurons"
adata = sc.read_h5ad(dir_path+"/{}.{}.LineMagnification.h5ad".format(DS,celltype))
def sanitize_strings(value):
    if isinstance(value, str):  # Only process string values
        return (
            value.replace('_', '')  # Replace underscores with spaces
            .replace('-', '')      # Replace hyphens with spaces
            .replace('@', '')       # Example: Remove "@" character
        )
    return value  # Leave non-strings untouched
pcs=15
n_neighbors=30
control = "DMSO"
prop = .2

colorDict = {"3PBA":["#FFFFFF","#9D9D9C","#1D1D1B"],
"DPHP":["#FFFFFF","#E3D5EA","#8F4A97"],
"BPA":["#FFFFFF","#D8AF59","#6B3010"],
"MEP":["#FFFFFF","#E1AED1","#B81A5D"],
"BPF":["#FFFFFF","#FFED00","#FBBA00"],
"MBzP":["#FFFFFF","#B9E0E4","#2C509F"],
"TCP":["#FFFFFF","#D3D800","#009640"],
"MIX":["#FFFFFF","#F28F83","#E52423"]}

sc.pl.umap(adata, color=["line","condition"], size=10, vmin='p1', vmax='p99')
# HVGs per line and contrast (agonist vs antagonist)

Compound = list(set(["_".join(i.split("_")[:-1]) for i in adata.obs["condition"].unique().tolist() if i != "DMSO"]+["DMSO"]))
for compound in Compound:
    adata.obs.loc[adata.obs["condition"].str.contains(compound),"compound"] = compound
adata.obs["compound"].value_counts().sum()
adata.obs["line_compound"] = adata.obs["line"].astype(str) + "_" + adata.obs["compound"].astype(str)
adata.obs["line_condition"] = adata.obs["line"].astype(str) + "_" + adata.obs["condition"].astype(str)
ReverseMappingDict = {sanitize_strings(i):i for i in  adata.obs["line_compound"].unique().tolist()}
ReverseMappingDict
HVGSCondict = {}
for compound in adata.obs["line_condition"].unique().tolist():
    adatacompound = adata[adata.obs["line_condition"] == compound]

    nsamples = len(adatacompound.obs["sample_id"].unique().tolist())
    print(f"Found {nsamples} repliacates for  {compound}")
    if nsamples < 3:
        minOverlap = nsamples
    else:
        minOverlap = nsamples-1
    print(f"Only HVGs common to {minOverlap} replicates will be kept")

    sc.pp.highly_variable_genes(adatacompound, batch_key="sample_id", flavor="seurat", n_top_genes=2000)
    CommonHVGs = set(adatacompound.var_names[adatacompound.var["highly_variable_nbatches"] >= minOverlap])
    nHVGs = len(CommonHVGs)
    print(f"Found {nHVGs} common HVGs across {minOverlap} replicates for  exposure {compound} ")
    print("\n")
    HVGSCondict[compound] = CommonHVGs
HVGSCondlineDict = {}
for compound in adata.obs["line_compound"].unique().tolist():

    adataCond = adata[adata.obs["line_compound"] == compound].copy()
    sc.pp.highly_variable_genes(adataCond, flavor="seurat", n_top_genes=2000)
    HVGSLine = set(adataCond.var_names[adataCond.var["highly_variable"]].tolist())
    nHVGs = len(HVGSLine)
    print(f"Found {nHVGs} common HVGs across for  exposure {compound} ")
    print("\n")
    HVGSCondlineDict[compound] = HVGSLine



IntersectionDEGs = {}
for k in [K for K in list(HVGSCondlineDict.keys()) if control not in K]:
    crossHVGs = HVGSCondlineDict[k]
    intraHVGs = HVGSCondict["{}_AGONIST".format(k)].union(HVGSCondict["{}_ANTAGONIST".format(k)])
    print("Found {} intra compound HVGs ".format(len(intraHVGs)))
    OverallHVGs = list(crossHVGs.union(intraHVGs))
    print("Found {} final HVGs for {}".format(len(OverallHVGs), k))
    IntersectionDEGs[k] = OverallHVGs

# Divide by line_compound and process each
IntegratedDict = {}
designDict = {}
for line_compound in [i for i in adata.obs["line_compound"].unique().tolist()if "DMSO" not in i]:
    print(f"Processing {line_compound}")
    adataCondition = adata[adata.obs["line_compound"] == line_compound].copy()
    print(adataCondition.shape)
    print(adataCondition.obs["line"].unique().tolist())
    print(adataCondition.obs["condition"].unique().tolist())
    adataCondition.var["highly_variable"] = adataCondition.var_names.isin(IntersectionDEGs[line_compound])
    print(adataCondition.var["highly_variable"].sum())

    sc.tl.pca(adataCondition, use_highly_variable=True)
    sc.pp.neighbors(adataCondition, n_pcs=pcs, n_neighbors=n_neighbors)
    sc.tl.umap(adataCondition)

    sc.pl.umap(adataCondition, color=["Consensus_call","replicate","condition"], size=40, vmin='p1', vmax='p99', ncols=3,wspace=.4)
    #IntegratedDict[condition] = adataCondition

    # And prepare each anndata for milo
    ParsedObs = adataCondition.obs[["sample_id","line_compound","line","condition","Consensus_call","replicate"]].copy()
    ParsedObs = ParsedObs.applymap(sanitize_strings).loc[adataCondition.obs_names]
    adataCondition.obs = ParsedObs.copy()
    adataCondition.obsp = None
    del adataCondition.uns["neighbors"]
    adataCondition.obs["MiloSlicer"] = sanitize_strings(line_compound)
    del adataCondition.layers
    del adataCondition.varm
    #Store anndata
    IntegratedDict[sanitize_strings(line_compound)] = adataCondition

    # And its design matrix
    design_df = adataCondition.obs.copy()
    design_df.drop_duplicates(inplace=True)
    design_df.index = design_df['sample_id']
    design_df.index.name = None
    print("The following design matrix will be used")
    print(design_df)
    design_df["MiloSlicer"] = sanitize_strings(line_compound)
    designDict[sanitize_strings(line_compound)] = design_df

CombinedDesign = pd.concat(list(designDict.values()))
CombinedAdatas = ad.concat(list(IntegratedDict.values()))

In [ ]:
tmp = sc.read_h5ad("/group/testa/Common/scRNAseq/visualization/ENDPOINTS_sc/{}.{}.LineMagnification.h5ad".format(DS,celltype))
sc.pl.umap(tmp, color=["line","condition"], size=10, vmin='p1', vmax='p99')

In [ ]:
colorDict = {'Depleted':"#534ca3", 'Enriched':"#be1622", 'NotEnriched':"white"}

for line_compound in list(IntegratedDict.keys()):
    adataCondition = IntegratedDict[line_compound]
    parsedContrast = (adataCondition.obs["MiloSlicer"].str.replace("CTL04E", "CTL04E_")\
        .str.replace("CTL08A", "CTL08A_")+"_AGONIST_DA").str.replace("ARYLHYD", "ARYL_HYD")\
            .str.replace("LIVERX", "LIVER-X").unique()[0]
    adataCondition.obs[parsedContrast] =  tmp.obs.loc[adataCondition.obs_names, parsedContrast]
    unscolors = [colorDict[cat] for cat in adataCondition.obs[parsedContrast].astype("category").cat.categories.tolist()]
    adataCondition.uns[f"{parsedContrast}_colors"] = unscolors
    sc.pl.umap(adataCondition, color=["Consensus_call","replicate","condition",parsedContrast], size=40, vmin='p1',
     vmax='p99', ncols=4,wspace=.6, save=f"_{line_compound}_withContrast.png", add_outline=True,outline_width=(0.1, 0.01))